# PawVision model training

Run the same modular pipeline as the command line, from the project root. Prepare the data first with `src/preprocessing.py`.


## Production model: transfer learning

The app uses a fine-tuned ImageNet backbone. Reproduce it from the project root:

```
python src/preprocessing.py --image-size 224 --destination data/processed_224
python src/train_transfer.py
python src/evaluate.py --promote --allow-repeat-test
```

The cells below train the original scratch CNN for comparison. They write to `outputs/experiments/notebook_scratch/` and never touch the production model.


In [ ]:
import sys
from pathlib import Path

import tensorflow as tf

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from model import build_model, make_dataset
from paths import DATASET_128, OUTPUTS

RUN_DIR = OUTPUTS / 'experiments' / 'notebook_scratch'
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
train_ds = make_dataset(str(DATASET_128 / 'train'), shuffle=True)
val_ds = make_dataset(str(DATASET_128 / 'validation'))
model = build_model()
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(RUN_DIR / 'best_model.keras', monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)


In [ ]:
# Final test scoring is done once, from the command line: python src/evaluate.py
# Evaluation artifacts and test metrics are written to outputs/.
